In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub

FILE_LOCATION = '/kaggle/input/competitions/playground-series-s6e8/'
train_dataset = pd.read_csv(FILE_LOCATION + 'train.csv')
test_dataset = pd.read_csv(FILE_LOCATION + 'test.csv')
TARGET = train_dataset['addicted_label']
train_dataset = train_dataset.drop(columns=['addicted_label', 'id','gender', 'academic_work_impact','age', 'stress_level'])
# train_dataset = train_dataset.drop(columns=['addicted_label', 'id'])
y_id = test_dataset['id']
# X_test = test_dataset.drop(columns=['id'])
X_test = test_dataset.drop(columns=['id','gender', 'academic_work_impact','age', 'stress_level'])

del test_dataset

/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e8/train.csv
/kaggle/input/competitions/playground-series-s6e8/test.csv


In [2]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    train_dataset, TARGET, 
    test_size=0.2, 
    random_state=42, 
    stratify=TARGET  # Keeps class proportions equal (use for classification)
)
del train_dataset

In [3]:
from sklearn.impute import SimpleImputer #for missing data

numeric_cols = X_train.select_dtypes(include='number').columns.tolist()

imputer_numeric = SimpleImputer(strategy='median')

# Fit and transform numeric columns
X_train[numeric_cols] = imputer_numeric.fit_transform(X_train[numeric_cols])
X_val[numeric_cols] = imputer_numeric.transform(X_val[numeric_cols])
X_test[numeric_cols] = imputer_numeric.transform(X_test[numeric_cols])

In [4]:
# %pip install xgboost
# import torch
# from xgboost import XGBClassifier
# from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

# use_device = 'cuda' if torch.cuda.is_available() else 'cpu'

# pos = (y_train == 1).sum()
# neg = (y_train == 0).sum()

# scale_pos_weight_value = neg / pos

# xgb_model = XGBClassifier(
#     missing=np.nan,# 1. Define what value represents missing data (default is np.nan)
#     enable_categorical=True, # 2. Enable native handling for pandas 'category' columns with NaNs
#     tree_method='hist',# Required for enable_categorical
#     random_state=48,
#     n_estimators=5000,
#     device=use_device,
#     max_depth=5,
#     eval_metric='logloss',
#     early_stopping_rounds=50,
#     learning_rate=0.05,
#     scale_pos_weight = scale_pos_weight_value # for class imbalance
# )
# # xgb_model.fit(X_train, y_train)
# xgb_model.fit(
#     X_train, y_train,
#     eval_set=[(X_val, y_val)],
#     verbose=False
# )
# print("Best iteration:", xgb_model.best_iteration)

# y_pred = xgb_model.predict(X_val)
# y_proba = xgb_model.predict_proba(X_val)[:, 1]

# # 4. Evaluate performance
# print("Accuracy:", accuracy_score(y_val, y_pred))
# print("ROC-AUC Score:", roc_auc_score(y_val, y_proba))
# print("\nClassification Report:\n", classification_report(y_val, y_pred))

In [5]:
%pip install xgboost
import torch
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

use_device = 'cuda' if torch.cuda.is_available() else 'cpu'

param_dist = {
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0, 0.1, 1, 5],
    'reg_lambda': [1, 5, 10, 20]
}

base_model = XGBClassifier(
    missing=np.nan,
    enable_categorical=True,
    tree_method='hist',
    device=use_device,          # reuse the same GPU/CPU detection from before
    random_state=48,
    eval_metric='logloss',
    n_estimators=5000,          # ceiling only — early stopping decides the real count
    early_stopping_rounds=50
)

xgb_model = RandomizedSearchCV(
    base_model,
    param_distributions=param_dist,
    n_iter=20,                                  # start conservative, raise later
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=48),
    random_state=48,
    n_jobs=1 if use_device == 'cuda' else -1,    # avoid GPU contention across folds
    verbose=1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

Note: you may need to restart the kernel to use updated packages.
Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [20:53:12] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


RandomizedSearchCV(cv=StratifiedKFold(n_splits=3, random_state=48, shuffle=True),
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device='cuda',
                                           early_stopping_rounds=50,
                                           enable_categorical=True,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,...
                                           n_estimators=5000, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=20, n_jobs=1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'learning_rate': [0.01, 0.03, 0.05,
                                                          0.08, 0.1],
                                        'max_depth': [3, 4, 5, 6, 8],
                                        'min_child_weight': [1, 3, 5, 7],
                                        'reg_alpha': [0, 0.1, 1, 5],
                                        'reg_lambda': [1, 5, 10, 20],
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   random_state=48, scoring='roc_auc', verbose=1)

In [6]:
print(xgb_model.best_params_)
print(xgb_model.best_score_)


y_pred = xgb_model.predict(X_val)
y_proba = xgb_model.predict_proba(X_val)[:, 1]

# 4. Evaluate performance
print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_val, y_proba))
print("\nClassification Report:\n", classification_report(y_val, y_pred))

{'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 0.1, 'min_child_weight': 7, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
0.9631799173024645
Accuracy: 0.9006827024603323
ROC-AUC Score: 0.9630148993494188

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.82      0.83     40179
           1       0.93      0.93      0.93     98095

    accuracy                           0.90    138274
   macro avg       0.88      0.88      0.88    138274
weighted avg       0.90      0.90      0.90    138274



In [7]:
# from sklearn.inspection import permutation_importance

# # Run permutation importance on the validation set
# perm_result = permutation_importance(
#     xgb_model,
#     X_val,
#     y_val,
#     scoring='roc_auc',      # match the metric you care about; use 'accuracy' if preferred
#     n_repeats=10,           # more repeats = more stable estimates, but slower
#     random_state=48,
#     n_jobs=-1
# )

# # Put results into a tidy, sorted DataFrame
# perm_importance_df = pd.DataFrame({
#     'feature': X_val.columns,
#     'importance_mean': perm_result.importances_mean,
#     'importance_std': perm_result.importances_std
# }).sort_values('importance_mean', ascending=False).reset_index(drop=True)

# print(perm_importance_df)

In [8]:
test_probabilities = xgb_model.predict_proba(X_test)[:, 1]

# Apply your custom threshold (replace 'best_threshold' with the exact float value 
# or variable name from your threshold-tuning step)
# This converts probabilities (e.g., 0.73) into hard classes (0 or 1)

# test_predictions = (test_probabilities >= best_threshold).astype(int)

submission = pd.DataFrame({
    'id': y_id,
    'addicted_label': test_probabilities
})
submission.to_csv('submission.csv', index=False)

print(submission.head())

       id  addicted_label
0  691369        0.999569
1  691370        0.959824
2  691371        0.923556
3  691372        0.989082
4  691373        0.999020
